In [0]:
dbutils.widgets.removeAll()

In [0]:
%sql
create widget text storageName default "adlsproyectoje";
create widget text catalogo default "catalog_au";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
%sql
DROP CATALOG IF EXISTS ${catalogo} CASCADE;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS ${catalogo}
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
%sql
DROP SCHEMA IF EXISTS ${catalogo}.raw;
DROP SCHEMA IF EXISTS ${catalogo}.bronze;
DROP SCHEMA IF EXISTS ${catalogo}.silver;
DROP SCHEMA IF EXISTS ${catalogo}.golden;

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS ${catalogo}.raw;
CREATE SCHEMA IF NOT EXISTS ${catalogo}.bronze;
CREATE SCHEMA IF NOT EXISTS ${catalogo}.silver;
CREATE SCHEMA IF NOT EXISTS ${catalogo}.golden;

###Tablas Bronze

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.bronze.circuits (
circuit_id integer,
circuit_ref string,
name string,
location string,
country string,
latitude double,
longitude double,
altitude integer,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/circuits"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.bronze.races (
race_id integer,
race_year integer,
round integer,
circuit_id integer,
name string,
ingestion_date timestamp
)
USING DELTA
PARTITIONED BY (race_year)
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/races"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.bronze.constructors (
  constructor_id integer,
  constructor_ref string,
  name string,
  nationality string,
  ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/constructors"

###Tablas Silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.silver.circuits_transformed (
  race_id integer,
  race_year integer,
  round integer,
  circuit_id integer,
  name_race string,
  ingestion_date timestamp,
  circuit_ref string,
  name string,
  location string,
  country string,
  latitude double,
  longitude double,
  altitude integer,
  altitude_category string,
  years_diferences integer,
  lat_diff integer,
  race_type string,
  near_equator string
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/circuits_transformed"

###Tablas Golden

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ${catalogo}.golden.golden_raced_partitioned (
  race_year integer,
  conteo long,
  max_altitude integer,
  min_altitude integer,
  country string,
  race_type string,
  near_equator string
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/golden_raced_partitioned"

POKEMON

#### Tabla ABILITIES
Catálogo de habilidades que pueden tener los Pokémon, identificadas por un ID único y su nombre.

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalogo}.bronze.ABILITIES (
    habilidad_id INT,
    nombre_habilidad STRING
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/ABILITIES";

#### Tabla EXPERIENCE_TYPES
Tipos de experiencia que determinan cómo los Pokémon ganan puntos de experiencia al subir de nivel.

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalogo}.bronze.EXPERIENCE_TYPES (
    tipo_experiencia_id INT,
    nombre_tipo_experiencia STRING
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/EXPERIENCE_TYPES";

#### Tabla GENERATIONS
Información sobre las generaciones de Pokémon, incluyendo número de generación y descripción.

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalogo}.bronze.GENERATIONS (
    generacion_id INT,
    numero_generacion INT,
    descripcion STRING
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/GENERATIONS";

#### Tabla POKEMON
Tabla principal con los datos de cada Pokémon: estadísticas base, dimensiones físicas, tipo, generación y características especiales (legendario, mega evolución, formas regionales).

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalogo}.bronze.POKEMON (
    pokemon_id INT,
    numero_pokedex INT,
    nombre STRING,
    tipo1_id INT,
    tipo2_id INT,
    generacion_id INT,
    tipo_experiencia_id INT,
    experiencia_nivel_100 INT,
    evolucion_final INT,
    tasa_captura INT,
    legendario INT,
    mega_evolucion INT,
    forma_alola INT,
    forma_galar INT,
    ps INT,
    ataque INT,
    defensa INT,
    ataque_especial INT,
    defensa_especial INT,
    velocidad INT,
    total_base INT,
    media DOUBLE,
    desviacion_estandar DOUBLE,
    altura_m DOUBLE,
    peso_kg DOUBLE,
    imc DOUBLE
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/POKEMON";

#### Tabla POKEMON_ABILITIES
Relación muchos a muchos entre Pokémon y sus habilidades.

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalogo}.bronze.POKEMON_ABILITIES (
    pokemon_id INT,
    habilidad_id INT
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/POKEMON_ABILITIES";

#### Tabla POKEMON_TYPE_EFFECTIVENESS
Efectividad de cada tipo atacante contra cada Pokémon, expresada como multiplicador de daño.

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalogo}.bronze.POKEMON_TYPE_EFFECTIVENESS (
    pokemon_id INT,
    tipo_atacante_id INT,
    multiplicador DOUBLE
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/POKEMON_TYPE_EFFECTIVENESS";

#### Tabla TYPES
Catálogo de tipos de Pokémon (fuego, agua, planta, etc.).

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalogo}.bronze.TYPES (
    tipo_id INT,
    nombre_tipo STRING
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/TYPES";